# Lesson 1: Guidelines for Prompting

DeepLearning.AI "ChatGPT Prompt Engineering for Developers" を Ollama 向けに変換。

## 学習目標
- 効果的なプロンプトを書く2つの原則を理解する
- 各原則のタクティクスを実践する

## 2つの原則
1. **明確で具体的な指示を書く**
2. **モデルに「考える時間」を与える**

## セットアップ

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model='qwen2.5-coder:14b')

def get_completion(prompt):
    """プロンプトを送信してテキストを返すヘルパー関数"""
    response = llm.invoke(prompt)
    return response.content

print('セットアップ完了')

---
## 原則1: 明確で具体的な指示を書く

### タクティクス1: 区切り文字を使う

区切り文字の例: ` ``` `, `"""`, `< >`, `<tag>`, `:`

区切り文字を使うことで、**プロンプトインジェクション**（入力テキストがモデルへの指示を上書きしてしまう攻撃）を防げる。

In [ ]:
text = f"""
あなたは、望む目標を達成するために、できる限り明確で
具体的な指示を提供することで、モデルを望む出力に
導く必要があります。これにより、モデルが何をすべきかを
明確に理解し、無関係または不正確な応答の可能性を減らせます。
明確なプロンプトを書くことと、短いプロンプトを書くことを
混同しないでください。多くの場合、長いプロンプトの方が
モデルにより多くの明確さとコンテキストを提供し、
より詳細で関連性の高い出力につながります。
"""

prompt = f"""
三重バッククォートで区切られたテキストを、1つの文に要約してください。
```{text}```
"""

response = get_completion(prompt)
print(response)

### タクティクス2: 構造化された出力を求める

JSON や HTML などの形式で出力を求めると、後処理が簡単になる。

In [ ]:
prompt = f"""
架空の本のタイトル、著者、ジャンルを含む3冊のリストを生成してください。
以下のキーを持つPython辞書のリストとしてJSON形式で提供してください:
book_id, title, author, genre
"""

response = get_completion(prompt)
print(response)

In [ ]:
# HTML 形式で出力
prompt = f"""
架空の本のタイトル、著者、ジャンルを含む3冊のリストを生成してください。
HTMLのテーブル形式で提供してください。
列は book_id, title, author, genre とします。
"""

response = get_completion(prompt)
print(response)

### タクティクス3: 条件が満たされているか確認する

In [ ]:
text_1 = f"""
お茶を一杯作るのは簡単です！まず、水を沸かす必要があります。
その間に、カップを取り出してティーバッグを入れてください。
水が十分熱くなったら、ティーバッグの上に注ぎます。
お茶を少し蒸らしましょう。数分後、ティーバッグを取り出します。
お好みで砂糖やミルクを加えてください。以上です！
"""

prompt = f"""
三重引用符で区切られたテキストが提供されます。
一連の指示が含まれている場合は、その指示を以下の形式で書き直してください:

ステップ1: ...
ステップ2: ...
...
ステップN: ...

テキストに一連の指示が含まれていない場合は、
「ステップが提供されていません。」とだけ書いてください。

\"\"\"{text_1}\"\"\"
"""

response = get_completion(prompt)
print('完了1:')
print(response)

In [ ]:
# 指示が含まれていないテキストの場合
text_2 = f"""
今日は素晴らしい天気です。鳥たちが歌い、
風もそよそよと心地よく吹いています。
公園に出かけてのんびり過ごしたい気分です。
"""

prompt = f"""
三重引用符で区切られたテキストが提供されます。
一連の指示が含まれている場合は、その指示を以下の形式で書き直してください:

ステップ1: ...
ステップ2: ...
...
ステップN: ...

テキストに一連の指示が含まれていない場合は、
「ステップが提供されていません。」とだけ書いてください。

\"\"\"{text_2}\"\"\"
"""

response = get_completion(prompt)
print('完了2:')
print(response)

### タクティクス4: Few-shot プロンプティング

タスクを実行する成功例を示してから、モデルに実行させる。

In [ ]:
prompt = f"""
あなたの仕事は、一貫したスタイルで答えることです。

<子供>: 忍耐について教えて。

<祖父母>: 最も深い谷を刻む川は、
控えめな泉から流れ出す。最も壮大な交響曲は、
一つの音符から生まれる。最も複雑なタペストリーは、
一本の孤独な糸から始まる。

<子供>: 回復力について教えて。
"""

response = get_completion(prompt)
print(response)

---
## 原則2: モデルに「考える時間」を与える

### タクティクス1: タスクを完了するためのステップを指定する

In [ ]:
text = f"""
チャーミングな村では、ジャックとジルという兄妹が
丘のてっぺんにある井戸から水を汲みに行く冒険に出かけます。
彼らが丘を登っていると、ジャックがつまずいて転び、
ジルもつられて転んでしまいました。
ちょっとした怪我をしましたが、二人は家に帰り、
温かく迎えられました。
この事故にもかかわらず、二人の冒険好きな精神は衰えず、
楽しく探索を続けました。
"""

prompt_1 = f"""
次のアクションを実行してください:
1 - 三重バッククォートで区切られたテキストを1文で要約する。
2 - 要約をフランス語に翻訳する。
3 - フランス語の要約の各名前を列挙する。
4 - 次のキーを含むJSONオブジェクトを出力する:
    french_summary, num_names

回答は改行で区切ってください。

テキスト:
```{text}```
"""

response = get_completion(prompt_1)
print('完了 - プロンプト1:')
print(response)

### タクティクス2: 結論を急がせず、自分で解を導かせる

In [ ]:
# 悪い例: モデルが生徒の誤った解答を鵜呑みにしてしまう
prompt = f"""
学生の解答が正しいかどうか判断してください。

質問:
太陽光発電設備を設置しています。
- 土地のコスト: 100ドル/平方フィート
- 太陽光パネルのコスト: 250ドル/平方フィート
- 年間メンテナンスコスト: 固定10万ドル + 10ドル/平方フィート

平方フィート数をxとして、初年度の総コストはいくら？

学生の解答:
初年度の総コスト = 土地コスト + パネルコスト + メンテナンスコスト
= 100x + 250x + 100,000 + 100x
= 450x + 100,000
"""

response = get_completion(prompt)
print(response)

In [ ]:
# 良い例: まず自分で解いてから、学生の解答と比較させる
prompt = f"""
あなたの仕事は、学生の解答が正しいかどうかを判断することです。
問題を解くには、次のことをしてください:
- まず、問題に対する自分自身の解を導き出す。
- 次に、自分の解と学生の解を比較し、学生の解が正しいかどうかを評価する。
- 自分で問題を解くまで、学生の解答が正しいかどうかを判断しないこと。

次の形式で出力してください:
自分の解: ```自分の解をここに```
学生の解は自分の解と同じか？: `はい`または`いいえ`
学生の評点: `正解`または`不正解`

質問:
太陽光発電設備を設置しています。
- 土地のコスト: 100ドル/平方フィート
- 太陽光パネルのコスト: 250ドル/平方フィート
- 年間メンテナンスコスト: 固定10万ドル + 10ドル/平方フィート

平方フィート数をxとして、初年度の総コストはいくら？

学生の解答:
初年度の総コスト = 100x + 250x + 100,000 + 100x = 450x + 100,000
"""

response = get_completion(prompt)
print(response)

---
## モデルの限界: ハルシネーション

モデルは存在しない情報を「もっともらしく」作り上げることがある（ハルシネーション）。

**対策**: 関連する引用を見つけてから回答するよう指示する。

In [ ]:
# ハルシネーションの例: 存在しない製品について尋ねる
prompt = f"""
AeroGlide UltraSlim Smart Toothbrush by Boie について教えてください。
"""

response = get_completion(prompt)
print(response)

---
## まとめ

| 原則 | タクティクス |
|---|---|
| 明確で具体的な指示 | ① 区切り文字を使う |
| | ② 構造化された出力を求める |
| | ③ 条件チェックを入れる |
| | ④ Few-shot で例を示す |
| 考える時間を与える | ① ステップを指定する |
| | ② 自分で解いてから比較させる |

次のレッスン → `02_iterative.ipynb`